In [1]:
import pandas as pd
from datetime import *
import matplotlib.pyplot as plt
import seaborn as sns
import os 
import plotly.graph_objects as go
from scipy.signal import find_peaks as fp

# Constants
global WORKING_DIRECTORY,RAWDATAFILES,SUPPORTED_FILEFORMATS,RAW_Data_DIR,MERGE_FILES

openpose_path =f"H:\\Documents\\My Training\\CAS\\CAS_DLS\\0_Module\\M4_Neural_Network\\project\\openpose"
OpenPose_Executable_prefix ="bin\\OpenPoseDemo.exe --video "

# Get the current working directory
cwd = os.getcwd()
WORKING_DIRECTORY = os.path.dirname(cwd) #Parent directory
print("The current working directory is:", WORKING_DIRECTORY)

DATA_DIRECTORY = os.path.join(WORKING_DIRECTORY,"Project\\Data")
print("The data directory is:", DATA_DIRECTORY)

RAW_Data_DIR=  os.path.join(DATA_DIRECTORY,"01-raw")
PREPROCESS_Data_DIR=  os.path.join(DATA_DIRECTORY,"02-preprocessed")
FEATURES_Data_DIR=  os.path.join(DATA_DIRECTORY,"03-features")
PREDICTIONS_Data_DIR=  os.path.join(DATA_DIRECTORY,"04-predictions")

The current working directory is: h:\Documents\My Training\CAS\CAS_DLS\0_Module\M4_Neural_Network
The data directory is: h:\Documents\My Training\CAS\CAS_DLS\0_Module\M4_Neural_Network\Project\Data


In [19]:
#Generate Batch Commands for Preprocessing
with open(openpose_path + "\\preprocess.bat", "w") as f:
    f.write("h:"+ "\n")
    f.write("cd \"" + openpose_path+ "\"\n")
    for folder in [f.path for f in os.scandir(RAW_Data_DIR) if f.is_dir()]:
        source = os.listdir(folder)
        filteredlist = [x for x in source if x.upper().split(".")[-1] in ["MP4"] ]
        for idx, file in enumerate(filteredlist):
            # if (file.find("Mag")!=-1):
            #     continue
            step=""
            if file.find("60fps")>-1:
            #--frame_step 2 for 60fps
                step ="--frame_step 2"
            
            sequence_folder = PREPROCESS_Data_DIR +"\\" + folder.split("\\")[-1]
            if not(os.path.exists(sequence_folder)):
                os.mkdir(sequence_folder)

            sequence_folder = PREPROCESS_Data_DIR +"\\" + folder.split("\\")[-1]+ "\\" +str(idx)
            if not(os.path.exists(sequence_folder)):
                os.mkdir(sequence_folder)
            line = " ".join([OpenPose_Executable_prefix, "\"" + os.path.join( folder,file) +"\"","--write_json", "\"" +sequence_folder+ "\"", step, "--number_people_max 1", "--display 0 --render_pose 0"]) 
            f.write(line+ "\n")
        


In [20]:
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

def read_folder(folder_path, class_idx):
    idx1, sub_folder_path = folder_path

    dfs = []
    print(sub_folder_path, "start")
    
    # for idx1, sequence_folder in enumerate([f for f in os.scandir(folder_path)]):

    #     print(sequence_folder)
    s_source = os.listdir(sub_folder_path)
    sequence_files = [x for x in s_source if x.upper().split(".")[-1] in ["JSON"] ]
    sequence_files.sort()

    for idx2, file in enumerate(sequence_files):
        print(file)
        # Open and read the JSON file
        with open(os.path.join(sub_folder_path, file), 'r') as timeframe:
            config_dict = json.load(timeframe)
            if file.find("60fps"):
                fps=30
            elif file.find("25fps"):
                fps=25
            elif file.find("24fps"):
                fps=24
            data = {"target":[class_idx],"sequence":[idx1],"timestep":[idx2],"fps":[fps]}
            if len(config_dict["people"]) >0:
                coord = list(config_dict["people"][0]["pose_keypoints_2d"])
                for i in range(int(len(coord) / 3)):

                    data["x" + str(i)]= [coord[i*3-3]]
                    data["y" + str(i)] = [coord[i*3-2]]
                    data["c" + str(i)]= [ coord[i*3-1]]

            new_df = pd.DataFrame.from_dict(data)
            dfs.append(new_df.copy())

    print(len(dfs))
    if dfs:
        print(sub_folder_path, "done")
        combined = pd.concat(dfs, ignore_index=True)
        combined["source_folder"] = sub_folder_path
        return combined
    
    print(sub_folder_path, "error")
    return None


#Make DataFrame with all sequences
results = []
for idx0, folder in enumerate( [f.path for f in os.scandir(PREPROCESS_Data_DIR) if f.is_dir()]):
    subfolders = [(idx1, f1) for idx1, f1 in enumerate( [f.path for f in os.scandir(folder) if f.is_dir()])]

    with ThreadPoolExecutor(max_workers=6) as executor:
         futures = [executor.submit(read_folder, sfolder,idx0) for sfolder in subfolders]
    
    for future in as_completed(futures):
            df = future.result()
            if df is not None:
                results.append(df)

final_df = pd.concat(results, ignore_index=True)

print(final_df.head())
print("Total rows:", len(final_df))


h:\Documents\My Training\CAS\CAS_DLS\0_Module\M4_Neural_Network\Project\Data\02-preprocessed\Bicep Curl\0 start
13559782-hd_1920_1080_60fps_000000000000_keypoints.json
h:\Documents\My Training\CAS\CAS_DLS\0_Module\M4_Neural_Network\Project\Data\02-preprocessed\Bicep Curl\1 start
h:\Documents\My Training\CAS\CAS_DLS\0_Module\M4_Neural_Network\Project\Data\02-preprocessed\Bicep Curl\10 start
13559782-hd_1920_1080_60fps_000000000002_keypoints.json
h:\Documents\My Training\CAS\CAS_DLS\0_Module\M4_Neural_Network\Project\Data\02-preprocessed\Bicep Curl\11 start
h:\Documents\My Training\CAS\CAS_DLS\0_Module\M4_Neural_Network\Project\Data\02-preprocessed\Bicep Curl\12 start
13559782-hd_1920_1080_60fps_000000000004_keypoints.json
h:\Documents\My Training\CAS\CAS_DLS\0_Module\M4_Neural_Network\Project\Data\02-preprocessed\Bicep Curl\13 start
13559782-hd_1920_1080_60fps_000000000006_keypoints.json
6547801-uhd_2160_3840_24fps_000000000000_keypoints.json
13559782-hd_1920_1080_60fps_000000000008_key

In [21]:
final_df

,target,sequence,timestep,fps,x0,y0,c0,x1,y1,c1,...,x22,y22,c22,x23,y23,c23,x24,y24,c24,source_folder
0,0,7,0,30,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,h:\Documents\My Training\CAS\CAS_DLS\0_Module\...
1,0,7,1,30,0.00,0.00,0.000000,960.055,397.838,0.799293,...,0.00,0.00,0.000000,0.00,0.00,0.000000,0.00,0.00,0.000000,h:\Documents\My Training\CAS\CAS_DLS\0_Module\...
2,0,7,2,30,0.00,0.00,0.000000,962.959,397.828,0.795862,...,0.00,0.00,0.000000,0.00,0.00,0.000000,0.00,0.00,0.000000,h:\Documents\My Training\CAS\CAS_DLS\0_Module\...
3,0,7,3,30,0.00,0.00,0.000000,965.882,397.793,0.799278,...,0.00,0.00,0.000000,0.00,0.00,0.000000,0.00,0.00,0.000000,h:\Documents\My Training\CAS\CAS_DLS\0_Module\...
4,0,7,4,30,0.00,0.00,0.000000,968.799,394.936,0.810000,...,0.00,0.00,0.000000,0.00,0.00,0.000000,0.00,0.00,0.000000,h:\Documents\My Training\CAS\CAS_DLS\0_Module\...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20903,3,8,491,30,1679.60,2666.07,0.688811,408.873,2404.940,0.595287,...,1588.28,2609.58,0.403481,1974.13,2677.31,0.759592,1917.53,2700.26,0.723655,h:\Documents\My Training\CAS\CAS_DLS\0_Module\...
20904,3,8,492,30,1679.42,2666.09,0.702630,420.306,2405.370,0.457201,...,1577.76,2609.49,0.369093,1974.11,2677.28,0.750183,1917.41,2700.31,0.717229,h:\Documents\My Training\CAS\CAS_DLS\0_Module\...
20905,3,8,493,30,1679.38,2666.07,0.716340,420.064,2405.140,0.658943,...,1588.48,2609.39,0.384022,1974.25,2677.27,0.750267,1917.34,2700.27,0.721548,h:\Documents\My Training\CAS\CAS_DLS\0_Module\...
20906,3,8,494,30,1679.35,2666.30,0.716610,408.697,2394.140,0.753870,...,1588.52,2609.54,0.384782,1974.15,2677.23,0.755365,1917.34,2700.26,0.725617,h:\Documents\My Training\CAS\CAS_DLS\0_Module\...


In [22]:
final_df.to_pickle(os.path.join(PREPROCESS_Data_DIR,"all_timesteps.pkl"))

In [2]:
import cv2
vidpath = os.path.join(RAW_Data_DIR, "Bicep Curl\\4367638-hd_1920_1080_30fps.mp4")
vid = cv2.VideoCapture(vidpath)

start = 0
end = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))
frame = start
every=1

while frame < end:
    success, img = vid.read()
    if success:
        img= cv2.resize(img,(200,200))
        if frame % every == 0:
            cv2.imwrite(os.path.join(FEATURES_Data_DIR,"test_image%d.jpg") %frame, img)
    frame += 1

vid=cv2.VideoCapture(0)
vid.release()